In [ ]:
%pip install --upgrade openai

In [ ]:
import os

from matplotlib import pyplot as plt

import image_tagger as it

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
input_dir = r"C:\Users\oloon\Dropbox\images\uploads"
output_dir = input_dir
metadata_filename = os.path.join(output_dir, "image_metadata.csv")
gallery_filename = os.path.join(output_dir, "index.html")

In [ ]:
# it.scramble_image_directory(input_dir, output_dir)

In [ ]:
filepaths = it.find_images(input_dir, metadata_filename=metadata_filename)
print("number of image files:", len(filepaths))
filepaths

In [ ]:
it.tag_images(filepaths, metadata_filename, verbose=2)

In [ ]:
import pandas as pd

metadata_df = pd.read_csv(metadata_filename)
print(metadata_df.shape)
print("n unique original filenames:", len(set(metadata_df["original_filename"])))
print("n unique clean filenames:", len(set(metadata_df["clean_filename"])))

# pd.set_option('display.max_rows', None)
# metadata_df[ metadata_df['clean_filename'].duplicated(keep=False) ].sort_values(['clean_filename'])[['original_filename', 'clean_filename']]

In [ ]:
it.rename_images(metadata_filename, verbose=1)

In [ ]:
it.generate_gallery(metadata_filename, gallery_filename)

In [ ]:
plt.figure(figsize=(8, 5))
metadata_df.groupby("category").size().plot.barh()
plt.title("Category Histogram")
plt.ylabel("Category")
plt.xlabel("Number of Images")

In [ ]:
plt.figure(figsize=(8, 10))
metadata_df.groupby("genre").size().plot.barh()
plt.title("Genre Histogram")
plt.ylabel("Genre")
plt.xlabel("Number of Images")

In [ ]:
plt.figure(figsize=(8, 10))
tag_counts = metadata_df["tags"].str.split(";").explode("tags").value_counts()
N_TAGS = 50
top_tag_counts = tag_counts[:N_TAGS][::-1]
top_tag_counts.plot.barh()

plt.xlabel("Number of Images")
plt.ylabel("Tags")
plt.title(f"Frequency of Top {N_TAGS} Tags")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.figure(figsize=(10, 5))
delta_seconds = np.array(
    pd.to_datetime(metadata_df["timestamp"]).diff().astype(np.int64) / 1e9
)

print("quartiles", np.quantile(delta_seconds, [0, 0.25, 0.5, 0.75, 1]))
bounds = np.quantile(delta_seconds, [0.05, 0.95])  # 90% CI
bounded = np.clip(delta_seconds, bounds[0], bounds[1])

plt.subplot(121)
plt.hist(bounded, bins=16, orientation="horizontal", rwidth=0.8)
plt.ylim(0, 8)
plt.ylabel("Response Time (Seconds)")
plt.xlabel("Number of Images")
plt.title("Response Time Histogram")

plt.subplot(122)
plt.plot(delta_seconds[1:], lw=1, alpha=0.7)
plt.ylim(0, 8)
median = np.median(delta_seconds)
plt.axhline(
    median, linestyle="dashed", label=f"Median {median:0.2f} s", color="red", lw=1
)
plt.legend(loc="lower right")
plt.xlabel("Image Index")
plt.title("Response Time");

In [ ]:
cost = metadata_df["total_tokens"].sum() * 0.0000025
cost